# Notebook 04 — Deep dive: confusion matrix

**Goal:** Understand the confusion matrix as a diagnostic tool.

After this notebook you will be able to:
- Read any confusion matrix and diagnose *where* a model fails
- Understand precision vs recall trade-offs per class
- Interpret normalised vs raw confusion matrices
- Identify which classes are systematically confused with each other

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.load_data import load_processed
from src.preprocess import get_X_y, ENDANGERMENT_ORDER
from src.train import split_data, load_model
from src.evaluate import evaluate_model
from src.visualise import plot_confusion_matrix

print('Imports OK')

In [ ]:
df = load_processed()
X, y = get_X_y(df)
X_train, X_test, y_train, y_test = split_data(X, y)

# Load the best model from notebook 03
# Change 'baseline' to 'class_weights' or 'smote' if you saved those
try:
    model = load_model('baseline')
    print('Loaded baseline model')
except FileNotFoundError:
    print('Run notebook 02 first to create the model.')

In [ ]:
results = evaluate_model(model, X_test, y_test, ENDANGERMENT_ORDER)
cm = results['cm']
labels = results['cm_labels']

print('Raw confusion matrix:')
print(pd.DataFrame(cm, index=labels, columns=labels).to_string())

In [ ]:
# ── What the diagonal tells you ───────────────────────────────────────
# Diagonal = number of correct predictions per class
print('=== CORRECT PREDICTIONS PER CLASS (diagonal) ===')
for i, label in enumerate(labels):
    correct = cm[i, i]
    total_true = cm[i].sum()
    recall = correct / total_true if total_true > 0 else 0
    print(f'  {label:<30}: {correct:>4} / {total_true:>4} correct  (recall = {recall:.0%})')

In [ ]:
# ── What the off-diagonal tells you ─────────────────────────────────
# Off-diagonal cells show WHERE the model is making mistakes.
# A high value at cm[i][j] means the model often predicts j when the truth is i.
print('=== TOP CONFUSIONS (off-diagonal) ===')
confusions = []
for i in range(len(labels)):
    for j in range(len(labels)):
        if i != j and cm[i, j] > 0:
            confusions.append((cm[i, j], labels[i], labels[j]))
confusions.sort(reverse=True)
for count, true_cls, pred_cls in confusions[:10]:
    print(f'  True: {true_cls:<30} → Predicted: {pred_cls:<30} ({count} times)')

In [ ]:
# ── Raw vs normalised — side by side ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

import seaborn as sns

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=axes[0])
axes[0].set_title('Raw counts')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')
plt.setp(axes[0].get_xticklabels(), rotation=30, ha='right')

# Normalised (recall)
row_sums = cm.sum(axis=1, keepdims=True)
cm_norm = np.where(row_sums > 0, cm / row_sums, 0)
sns.heatmap(cm_norm, annot=True, fmt='.0%', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=axes[1])
axes[1].set_title('Normalised (recall per class)')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')
plt.setp(axes[1].get_xticklabels(), rotation=30, ha='right')

plt.suptitle('Confusion matrix: raw vs normalised', fontsize=14)
plt.tight_layout()
plt.savefig('../outputs/figures/04_cm_side_by_side.png', dpi=150)
plt.show()

print("""
RAW (left):   Shows absolute sample counts. Easy to see where most errors are.
NORMALISED (right): Shows what % of each TRUE class was correctly predicted.
              Compare rows: a class with 10% correct is being mostly ignored.
"""
)

In [ ]:
print('=== WHAT YOU HAVE LEARNED ===')
print()
print('Confusion matrix:')
print('  - Rows = true label, Columns = predicted label')
print('  - Diagonal = correct, Off-diagonal = mistakes')
print('  - Normalised version shows recall per class')
print()
print('Class imbalance symptoms visible in the confusion matrix:')
print('  - Minority class rows are mostly zeros (model never predicts them)')
print('  - Off-diagonal values concentrate in majority class columns')
print()
print('You now have the tools to diagnose ANY classifier on ANY imbalanced dataset.')